# VLM

Загружаем весь датасет `vsevolod-nv/aiconf-butterfly-detection-goldenset-extended`,
скачиваем картинки по URL в локальную папку `downloads`, прогоняем модель и считаем
базовые метрики: `precision`, `recall` и `mean IoU`.

Если нужен быстрый smoke-test, можно временно задать `MAX_ROWS`.

In [12]:
import base64
import json
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
import requests
from datasets import load_dataset
from dotenv import load_dotenv
from IPython.display import display
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

load_dotenv()

SOY_TOKEN = os.getenv("SOY_TOKEN")
if not SOY_TOKEN:
    raise ValueError("SOY_TOKEN не найден в .env")

NOTEBOOK_DIR = Path(".")
DOWNLOADS_DIR = NOTEBOOK_DIR / "downloads"
DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "vsevolod-nv/aiconf-butterfly-detection-goldenset-extended"
MAX_ROWS = None
IOU_THRESHOLD = 0.5
MODEL_URL = "https://api.eliza.yandex.net/internal/qwen3-5-397b-a17b-fp8/v1/chat/completions"

PROMPT = """
Ты — модель для разметки изображений. Твоя задача: по входной картинке найти бабочку, если она есть, и вернуть координаты прямоугольника в формате, совместимом с примером из TSV-файла.

Правила разметки:
1. Выделяй только бабочек.
2. Если на изображении нет бабочки, верни пустой ответ:
{}
3. Прямоугольник должен плотно охватывать всю видимую бабочку:
   - включай крылья,
   - включай усики,
   - включай лапки,
   - не обрезай части тела,
   - лучше сделать рамку чуть больше, чем чуть меньше.
4. Если бабочка частично скрыта, выделяй только видимую часть.
5. Не выделяй:
   - тень бабочки,
   - фон,
   - листья, цветы, ветки, кору и другие объекты,
   - других насекомых, если это не бабочка.
6. Если фон сложный, пестрый, бабочка маскируется или почти сливается с окружением — все равно постарайся найти и выделить ее.
7. Если в кадре несколько бабочек, верни прямоугольник для каждой бабочки отдельным объектом в массиве.
8. Координаты должны быть нормализованными числами от 0 до 1 относительно ширины и высоты изображения.
9. Формат полей:
   - \"shape\": всегда \"rectangle\"
   - \"left\": x-координата левого края
   - \"top\": y-координата верхнего края
   - \"width\": ширина прямоугольника
   - \"height\": высота прямоугольника
10. ОБЯЗАТЕЛЬНО выделить ВСЮ бабочку ЦЕЛИКОМ.

Формат ответа:
- Если бабочка одна:
{\"shape\":\"rectangle\",\"left\":0.123,\"top\":0.456,\"width\":0.321,\"height\":0.222}

- Если бабочек несколько:
[
  {\"shape\":\"rectangle\",\"left\":0.123,\"top\":0.456,\"width\":0.321,\"height\":0.222},
  {\"shape\":\"rectangle\",\"left\":0.654,\"top\":0.111,\"width\":0.120,\"height\":0.180}
]

- Если бабочки нет:
{}

Требования к ответу:
- Возвращай только JSON.
- Без пояснений, без markdown, без комментариев.
- Не добавляй никаких полей кроме: shape, left, top, width, height.
- Все числа должны быть десятичными.
- Не округляй слишком грубо: старайся давать точные координаты.

Сначала проанализируй изображение, потом верни результат.
{{image}}
""".strip()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)

print("Dataset:", DATASET_NAME)
print("Downloads dir:", DOWNLOADS_DIR.resolve())
print("MAX_ROWS:", MAX_ROWS)
print("IoU threshold:", IOU_THRESHOLD)

Dataset: vsevolod-nv/aiconf-butterfly-detection-goldenset-extended
Downloads dir: /Users/eraserhead/arcadia/junk/eraserhead/aiconf/model-runs/downloads
MAX_ROWS: None
IoU threshold: 0.5


In [13]:
dataset_dict = load_dataset(DATASET_NAME)
split_names = list(dataset_dict.keys())

frames = []
for split_name in split_names:
    split_df = dataset_dict[split_name].to_pandas()
    split_df["dataset_split"] = split_name
    frames.append(split_df)

data_df = pd.concat(frames, ignore_index=True)
if MAX_ROWS is not None:
    data_df = data_df.head(MAX_ROWS).copy()

print("Splits:", split_names)
print("Rows:", len(data_df))
display(data_df.head())

Splits: ['train']
Rows: 356


,photo_id,image,entity,bbox,dataset_split
0,624311177,https://inaturalist-open-data.s3.amazonaws.com/photos/624311177/original.jpg,beetle,None,train
1,623396315,https://inaturalist-open-data.s3.amazonaws.com/photos/623396315/original.jpg,butterfly,"{'shape': 'rectangle', 'left': 0.2896379526, 'top': 0.2420411985, 'width': 0.557636288, 'height': 0.5909280067}",train
2,624964698,https://inaturalist-open-data.s3.amazonaws.com/photos/624964698/original.jpg,beetle,None,train
3,623186705,https://inaturalist-open-data.s3.amazonaws.com/photos/623186705/original.jpg,bee,None,train
4,624510821,https://inaturalist-open-data.s3.amazonaws.com/photos/624510821/original.jpg,butterfly,"{'shape': 'rectangle', 'left': 0.3139006953, 'top': 0.203755722, 'width': 0.2904001085, 'height': 0.4494382022}",train


In [14]:
def image_filename(row) -> str:
    image_url = str(row["image"])
    suffix = Path(urlparse(image_url).path).suffix or ".jpg"
    return f"{row['photo_id']}{suffix}"


def download_image(image_url: str, target_path: Path) -> Path:
    if not target_path.exists():
        response = requests.get(image_url, timeout=120)
        response.raise_for_status()
        target_path.write_bytes(response.content)
    return target_path


data_df = data_df.copy()
local_paths = []
for row in tqdm(data_df.to_dict("records"), total=len(data_df), desc="download"):
    image_url = str(row["image"])
    local_path = DOWNLOADS_DIR / image_filename(row)
    download_image(image_url, local_path)
    local_paths.append(str(local_path))

data_df["local_image"] = local_paths
display(data_df[["photo_id", "image", "local_image"]].head())

download: 100%|██████████| 356/356 [11:25<00:00,  1.92s/it]


,photo_id,image,local_image
0,624311177,https://inaturalist-open-data.s3.amazonaws.com/photos/624311177/original.jpg,downloads/624311177.jpg
1,623396315,https://inaturalist-open-data.s3.amazonaws.com/photos/623396315/original.jpg,downloads/623396315.jpg
2,624964698,https://inaturalist-open-data.s3.amazonaws.com/photos/624964698/original.jpg,downloads/624964698.jpg
3,623186705,https://inaturalist-open-data.s3.amazonaws.com/photos/623186705/original.jpg,downloads/623186705.jpg
4,624510821,https://inaturalist-open-data.s3.amazonaws.com/photos/624510821/original.jpg,downloads/624510821.jpg


In [15]:
def encode_image_file(image_path: str) -> str:
    return base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")


def ask_model_with_image(prompt: str, image_b64: str) -> dict:
    headers = {
        "Authorization": f"OAuth {SOY_TOKEN}",
        "Content-Type": "application/json",
    }

    prompt_parts = prompt.split("{{image}}")
    content = []

    if prompt_parts[0]:
        content.append({"type": "text", "text": prompt_parts[0]})

    content.append(
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
        }
    )

    for part in prompt_parts[1:]:
        if part:
            content.append({"type": "text", "text": part})

    payload = {
        "messages": [
            {
                "role": "user",
                "content": content,
            }
        ]
    }

    response = requests.post(
        url=MODEL_URL,
        json=payload,
        headers=headers,
        timeout=310,
        verify=False,
    )
    response.raise_for_status()
    return response.json()


def extract_text_response(response_json: dict) -> str:
    try:
        return response_json["response"]["choices"][0]["message"]["content"].strip()
    except Exception as exc:
        raise ValueError(
            "Не удалось извлечь текст ответа модели:\n"
            f"{exc}\n{json.dumps(response_json, ensure_ascii=False, indent=2)}"
        ) from exc


def extract_json_block(text: str):
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    md_match = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, re.DOTALL)
    if md_match:
        return json.loads(md_match.group(1))

    obj_match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if obj_match:
        return json.loads(obj_match.group(1))

    raise ValueError(f"Не удалось найти JSON в ответе модели:\n{text}")


def normalize_boxes(parsed):
    if parsed in ({}, None):
        return []
    if isinstance(parsed, dict):
        return [parsed]
    if isinstance(parsed, list):
        return parsed
    raise ValueError(f"Неожиданный формат ответа: {type(parsed)}")


def coerce_bbox_list(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, dict):
        if value.get("shape") == "rectangle":
            return [value]
        if "bbox" in value:
            return coerce_bbox_list(value["bbox"])
    if isinstance(value, list):
        boxes = []
        for item in value:
            boxes.extend(coerce_bbox_list(item))
        return boxes
    return []


def get_gt_boxes(row: dict):
    for key in ["bbox", "bboxes", "boxes", "result", "annotations", "label"]:
        if key in row:
            boxes = coerce_bbox_list(row[key])
            if boxes or row[key] is None:
                return boxes
    return []


def clamp_box(box: dict) -> dict:
    left = max(0.0, min(1.0, float(box["left"])))
    top = max(0.0, min(1.0, float(box["top"])))
    width = max(0.0, min(1.0 - left, float(box["width"])))
    height = max(0.0, min(1.0 - top, float(box["height"])))
    return {
        "shape": "rectangle",
        "left": left,
        "top": top,
        "width": width,
        "height": height,
    }


def box_to_xyxy(box: dict):
    box = clamp_box(box)
    x1 = box["left"]
    y1 = box["top"]
    x2 = x1 + box["width"]
    y2 = y1 + box["height"]
    return x1, y1, x2, y2


def compute_iou(box_a: dict, box_b: dict) -> float:
    ax1, ay1, ax2, ay2 = box_to_xyxy(box_a)
    bx1, by1, bx2, by2 = box_to_xyxy(box_b)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter_area
    if union <= 0:
        return 0.0
    return inter_area / union


def greedy_match(gt_boxes, pred_boxes, iou_threshold=0.5):
    candidates = []
    for gt_idx, gt_box in enumerate(gt_boxes):
        for pred_idx, pred_box in enumerate(pred_boxes):
            iou = compute_iou(gt_box, pred_box)
            if iou >= iou_threshold:
                candidates.append((iou, gt_idx, pred_idx))

    candidates.sort(reverse=True)
    used_gt = set()
    used_pred = set()
    matches = []

    for iou, gt_idx, pred_idx in candidates:
        if gt_idx in used_gt or pred_idx in used_pred:
            continue
        used_gt.add(gt_idx)
        used_pred.add(pred_idx)
        matches.append({"gt_idx": gt_idx, "pred_idx": pred_idx, "iou": iou})

    return matches


def evaluate_detection(gt_boxes, pred_boxes, iou_threshold=0.5):
    gt_boxes = [clamp_box(box) for box in gt_boxes]
    pred_boxes = [clamp_box(box) for box in pred_boxes]
    matches = greedy_match(gt_boxes, pred_boxes, iou_threshold=iou_threshold)
    tp = len(matches)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    mean_iou = sum(match["iou"] for match in matches) / tp if tp else 0.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "matched_ious": [match["iou"] for match in matches],
        "mean_iou": mean_iou,
    }


def draw_boxes(image: Image.Image, boxes, color="red"):
    image = image.copy().convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for box in boxes:
        box = clamp_box(box)
        x1 = box["left"] * width
        y1 = box["top"] * height
        x2 = (box["left"] + box["width"]) * width
        y2 = (box["top"] + box["height"]) * height
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
    return image

In [17]:
def predict_row(row: dict) -> dict:
    gt_boxes = get_gt_boxes(row)
    local_image = row["local_image"]

    try:
        image_b64 = encode_image_file(local_image)
        response_json = ask_model_with_image(PROMPT, image_b64)
        raw_text = extract_text_response(response_json)
        parsed = extract_json_block(raw_text)
        pred_boxes = normalize_boxes(parsed)
        error = None
    except Exception as exc:
        raw_text = ""
        pred_boxes = []
        error = repr(exc)

    metrics = evaluate_detection(gt_boxes, pred_boxes, iou_threshold=IOU_THRESHOLD)

    return {
        "photo_id": row.get("photo_id"),
        "dataset_split": row.get("dataset_split"),
        "entity": row.get("entity"),
        "image": row.get("image"),
        "local_image": local_image,
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "raw_text": raw_text,
        "error": error,
        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "mean_iou": metrics["mean_iou"],
        "matched_ious": metrics["matched_ious"],
    }


records = data_df.to_dict("records")
results = []
for row in tqdm(records, total=len(records), desc="inference"):
    results.append(predict_row(row))

results_df = pd.DataFrame(results)
results_df.head()

inference:   0%|          | 0/356 [00:00<?, ?it/s]/Users/eraserhead/arcadia/junk/eraserhead/aiconf/aiconf/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.eliza.yandex.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
inference:   0%|          | 1/356 [00:01<08:18,  1.40s/it]/Users/eraserhead/arcadia/junk/eraserhead/aiconf/aiconf/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.eliza.yandex.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
inference:   1%|          | 2/356 [00:02<06:03,  1.03s/it]/Users/eraserhead/arcadia/junk/eraserhead/aiconf/aiconf/lib/python3.13/site-packages/urllib3/connectionpool.py:1

,photo_id,dataset_split,entity,image,local_image,gt_boxes,pred_boxes,raw_text,error,tp,fp,fn,mean_iou,matched_ious
0,624311177,train,beetle,https://inaturalist-open-data.s3.amazonaws.com/photos/624311177/original.jpg,downloads/624311177.jpg,[],[],{},None,0,0,0,0.000000,[]
1,623396315,train,butterfly,https://inaturalist-open-data.s3.amazonaws.com/photos/623396315/original.jpg,downloads/623396315.jpg,"[{'shape': 'rectangle', 'left': 0.2896379526, 'top': 0.2420411985, 'width': 0.557636288, 'height': 0.5909280067}]","[{'shape': 'rectangle', 'left': 0.305, 'top': 0.278, 'width': 0.586, 'height': 0.578}]","{""shape"":""rectangle"",""left"":0.305,""top"":0.278,""width"":0.586,""height"":0.578}",None,1,0,0,0.819378,[0.8193778128042788]
2,624964698,train,beetle,https://inaturalist-open-data.s3.amazonaws.com/photos/624964698/original.jpg,downloads/624964698.jpg,[],[],{},None,0,0,0,0.000000,[]
3,623186705,train,bee,https://inaturalist-open-data.s3.amazonaws.com/photos/623186705/original.jpg,downloads/623186705.jpg,[],[],{},None,0,0,0,0.000000,[]
4,624510821,train,butterfly,https://inaturalist-open-data.s3.amazonaws.com/photos/624510821/original.jpg,downloads/624510821.jpg,"[{'shape': 'rectangle', 'left': 0.3139006953, 'top': 0.203755722, 'width': 0.2904001085, 'height': 0.4494382022}]",[],{},None,0,0,1,0.000000,[]


In [19]:
total_tp = int(results_df["tp"].sum())
total_fp = int(results_df["fp"].sum())
total_fn = int(results_df["fn"].sum())

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
matched_ious = [iou for values in results_df["matched_ious"] for iou in values]
mean_iou = sum(matched_ious) / len(matched_ious) if matched_ious else 0.0
error_count = int(results_df["error"].notna().sum())

metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": precision},
        {"metric": "recall", "value": recall},
        {"metric": "mean_iou", "value": mean_iou},
        {"metric": "tp", "value": total_tp},
        {"metric": "fp", "value": total_fp},
        {"metric": "fn", "value": total_fn},
        {"metric": "errors", "value": error_count},
        {"metric": "rows", "value": len(results_df)},
    ]
)

display(metrics_df)
display(results_df[["photo_id", "dataset_split", "entity", "tp", "fp", "fn", "mean_iou", "error"]].head(20))

error_rows = results_df.loc[results_df["error"].notna(), ["photo_id", "dataset_split", "entity", "error"]]
if not error_rows.empty:
    print("\nErrors:")
    for row in error_rows.itertuples(index=False):
        print(f"photo_id={row.photo_id} split={row.dataset_split} entity={row.entity}")
        print(row.error)
        print("-" * 120)


,metric,value
0,precision,0.896739
1,recall,0.926966
2,mean_iou,0.817233
3,tp,165.000000
4,fp,19.000000
5,fn,13.000000
6,errors,0.000000
7,rows,356.000000


,photo_id,dataset_split,entity,tp,fp,fn,mean_iou,error
0,624311177,train,beetle,0,0,0,0.000000,None
1,623396315,train,butterfly,1,0,0,0.819378,None
2,624964698,train,beetle,0,0,0,0.000000,None
3,623186705,train,bee,0,0,0,0.000000,None
4,624510821,train,butterfly,0,0,1,0.000000,None
5,591604963,train,shrub,0,0,0,0.000000,None
6,620547334,train,butterfly,1,0,0,0.788144,None
7,624601796,train,butterfly,1,0,0,0.867734,None
8,624594045,train,butterfly,1,0,0,0.810858,None
9,624963385,train,flower,0,0,0,0.000000,None


In [ ]:
from datetime import datetime, timezone
from tempfile import TemporaryDirectory

from huggingface_hub import HfApi

REPO_ID = "vsevolod-nv/aiconf-butterfly-qwen-eval"

upload_df = results_df.copy()
upload_df = upload_df.drop(columns=["local_image"], errors="ignore")

readme = f"""---
pretty_name: aiconf-butterfly-qwen-eval
tags:
- butterflies
- vlm
- object-detection
- evaluation
---

Qwen evaluation results for butterfly detection.

Generated at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}
Rows: {len(upload_df)}
"""

api = HfApi()

with TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    data_path = tmp_path / "results.jsonl"
    readme_path = tmp_path / "README.md"

    upload_df.to_json(data_path, orient="records", lines=True, force_ascii=False)
    readme_path.write_text(readme, encoding="utf-8")

    api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True)
    api.upload_folder(
        repo_id=REPO_ID,
        repo_type="dataset",
        folder_path=str(tmp_path),
        commit_message="Upload qwen eval results",
    )

print(f"Uploaded to: https://huggingface.co/datasets/{REPO_ID}")

Saved: /Users/eraserhead/arcadia/junk/eraserhead/aiconf/model-runs/eval_goldenset_extended_results.csv
